# IMP-zoom
the baseline downsizes each page to about 1024 visual tokens, which blurs small text. Instead of doubling the budget (IMP-hires), cut the retrieved page into native-resolution tiles, let ColSmol pick the tile most relevant to the question, and give the generator that tile (plus the full page for context).

- `IMP-zoom-top1`: best tile + full page of the VisRet top-1 page (about 2048 tokens max, same as IMP-hires).
- `IMP-zoomonly-top1`: best tile only (about 1024 tokens, same as baseline).
- `IMP-zoom-oracle`: best tile + full page of the gold page (isolates tile selection from retrieval).

In [ ]:
!pip install -q datasets huggingface_hub rank_bm25 sentence-transformers
!pip install -q transformers accelerate qwen-vl-utils colpali-engine

In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 84.4 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## 1. Config and data

In [ ]:
import os, re, json, string, random, math, time, glob
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

SET_NAME = "InfoVQA"            # rerun the whole notebook with "InfoVQA"
REPO = {"MP-DocVQA": "openbmb/VisRAG-Ret-Test-MP-DocVQA",
        "InfoVQA":   "openbmb/VisRAG-Ret-Test-InfoVQA"}[SET_NAME]
CACHE = f"/content/drive/MyDrive/visrag/{SET_NAME}"
os.makedirs(CACHE, exist_ok=True)
N_QUESTIONS, SEED = 200, 0

# ONE prompt for ALL systems
PROMPT = ("Answer the question using the provided document. "
          "Reply with only the short answer, no explanation.\n"
          "Question: {q}")

Mounted at /content/drive


In [ ]:
import csv
from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download

def load_beir_qrels(path):
    qrels = {}
    with open(path) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            qrels.setdefault(row["query-id"], {})[row["corpus-id"]] = int(row["score"])
    return qrels

corpus  = load_dataset(REPO, name="corpus",  split="train")
queries = load_dataset(REPO, name="queries", split="train")
qf = [f for f in list_repo_files(REPO, repo_type="dataset")
      if f.startswith("qrels/") and f.endswith(".tsv")]
assert len(qf) == 1, qf
qrels_raw = load_beir_qrels(hf_hub_download(REPO, qf[0], repo_type="dataset"))

print(corpus[0].keys(), queries[0])
page_ids = [str(r["corpus-id"]) for r in corpus]
qrels = {qid: {pid for pid, s in d.items() if s > 0} for qid, d in qrels_raw.items()}
q_all = [{"qid": str(r["query-id"]), "question": r["query"],
          "answers": r["answer"] if isinstance(r["answer"], list) else [r["answer"]]}
         for r in queries]

sub_path = f"{CACHE}/subset_qids.json"
if os.path.exists(sub_path):
    keep = set(json.load(open(sub_path)))
else:
    random.seed(SEED)
    keep = {q["qid"] for q in random.sample(q_all, min(N_QUESTIONS, len(q_all)))}
    json.dump(sorted(keep), open(sub_path, "w"))
q_sub = [q for q in q_all if q["qid"] in keep]
print(len(page_ids), "pages |", len(q_all), "questions |", len(q_sub), "in subset")
print("qrels pages found in corpus:", len({p for s in qrels.values() for p in s} & set(page_ids)))

README.md:   0%|          | 0.00/2.05k [00:00<?, ?B/s]

corpus/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  181MB            

corpus/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/459 [00:00<?, ? examples/s]

queries/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 42.0kB            

queries/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/718 [00:00<?, ? examples/s]

infographicsvqa-eval-qrels.tsv:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

dict_keys(['corpus-id', 'image']) {'query-id': '36966.jpeg-1', 'query': 'Which social media site was invented in 2002?', 'answer': ['friendster'], 'options': None, 'is_numerical': 0}
459 pages | 718 questions | 200 in subset
qrels pages found in corpus: 459


In [ ]:
import torch, gc, math, string, re
page_idx = {p: i for i, p in enumerate(page_ids)}
ocr_text = json.load(open(f"{CACHE}/ocr.json"))
text_rank = json.load(open(f"{CACHE}/best_text_rank.json"))
sub_ids = [q["qid"] for q in q_sub]
all_ids = [q["qid"] for q in q_all]

def retrieval_metrics(rank, qrels, qids):
    r1 = r5 = mrr = ndcg = 0.0
    for qid in qids:
        rel, top = qrels[qid], rank[qid]
        r1 += any(d in rel for d in top[:1])
        r5 += any(d in rel for d in top[:5])
        for k, d in enumerate(top[:10]):
            if d in rel:
                mrr += 1 / (k + 1); break
        dcg  = sum(1 / math.log2(k + 2) for k, d in enumerate(top[:5]) if d in rel)
        idcg = sum(1 / math.log2(k + 2) for k in range(min(len(rel), 5)))
        ndcg += dcg / idcg
    n = len(qids)
    return {"Recall@1": r1/n, "Recall@5": r5/n, "MRR@10": mrr/n, "nDCG@5": ndcg/n}

def norm(s):
    s = s.lower()
    s = "".join(c for c in s if c not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def correct(pred, golds):
    return norm(pred) in {norm(g) for g in golds}

def page_image(pid): return corpus[page_idx[pid]]["image"].convert("RGB")

In [ ]:
vis_rank = json.load(open(f"{CACHE}/visret_rank.json"))
gold_page = lambda q: sorted(qrels[q["qid"]])[0]
def load_preds(tag): return json.load(open(f"{CACHE}/gen_{tag}.json"))
print("cached systems:", sorted(os.path.basename(f)[4:-5] for f in glob.glob(f"{CACHE}/gen_*.json")))

cached systems: ['IMP-hires-top1', 'IMP-zoom-oracle', 'IMP-zoom-top1', 'IMP-zoomonly-top1', 'NoRetrieval', 'Oracle-image', 'Oracle-image-hires', 'Oracle-text', 'TextRAG-top1', 'TextRAG-top3', 'VisRAG-top1', 'VisRAG-top3']


## 2. Generator (baseline settings) and ColSmol (tile selector)

In [ ]:
def load_generator():
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    name = "Qwen/Qwen2.5-VL-3B-Instruct"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        name, torch_dtype=torch.float16, device_map="auto")      # T4: fp16, not bf16
    proc = AutoProcessor.from_pretrained(
        name, min_pixels=256*28*28, max_pixels=1024*28*28)       # lower max_pixels if OOM
    return model, proc

def generate(model, proc, question, texts=None, images=None, max_new_tokens=32):
    content = []
    for t in (texts or []):
        content.append({"type": "text", "text": f"Document:\n{t}"})
    for im in (images or []):
        content.append({"type": "image", "image": im})
    content.append({"type": "text", "text": PROMPT.format(q=question)})
    prompt = proc.apply_chat_template([{"role": "user", "content": content}],
                                      tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[prompt], images=images or None, return_tensors="pt",
                  padding=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs.input_ids.shape[1]:],
                             skip_special_tokens=True)[0].strip()

def run_system(model, proc, tag, build_inputs):
    """Generic runner: build_inputs(q) -> dict(texts=..., images=...). Cached per question."""
    path = f"{CACHE}/gen_{tag}.json"
    preds = json.load(open(path)) if os.path.exists(path) else {}
    for q in q_sub:
        if q["qid"] in preds: continue
        preds[q["qid"]] = generate(model, proc, q["question"], **build_inputs(q))
        if len(preds) % 20 == 0: json.dump(preds, open(path, "w"))
    json.dump(preds, open(path, "w"))
    acc = float(np.mean([correct(preds[q["qid"]], q["answers"]) for q in q_sub]))
    print(f"{tag}: accuracy = {acc:.3f}")
    return preds, acc

In [ ]:
# Load the generator first
model, proc = load_generator()

# Perform all required generations with the generator model before freeing it
# (We will load ColSmol later in the notebook when we run task 4 zoom setups)
print("Generator loaded successfully.")

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

... (3 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Generator loaded successfully.


In [ ]:
model, proc = load_generator()

from colpali_engine.models import ColIdefics3, ColIdefics3Processor
VNAME = "vidore/colSmol-500M"
vmodel = ColIdefics3.from_pretrained(VNAME, torch_dtype=torch.float16, device_map="cuda").eval()
vproc = ColIdefics3Processor.from_pretrained(VNAME)

def trim(e): return e[e.abs().sum(-1) > 0].cpu()

@torch.no_grad()
def embed_images(imgs, bs=4):
    out = []
    for b in range(0, len(imgs), bs):
        batch = vproc.process_images(imgs[b:b+bs]).to(vmodel.device)
        out.extend(trim(x) for x in vmodel(**batch))
    return out

@torch.no_grad()
def embed_questions(qs, bs=16):
    out = {}
    for b in range(0, len(qs), bs):
        chunk = qs[b:b+bs]
        batch = vproc.process_queries([q["question"] for q in chunk]).to(vmodel.device)
        for q, e in zip(chunk, vmodel(**batch)): out[q["qid"]] = trim(e)
    return out

q_emb = embed_questions(q_sub)
print("query embeddings:", len(q_emb))

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.62k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  921MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/490 [00:00<?, ?it/s]

... (10 outputs left out)

added_tokens.json:   0%|          | 0.00/4.74k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

query embeddings: 200


## 3. Tiling and tile selection
Tiles are windows of at most the baseline pixel budget (`1024*28*28`) cut from the original page with 15% overlap, so each tile is shown to the generator at about native resolution. ColSmol scores each tile against the question (MaxSim) and the top tile is chosen. Choices are cached on Drive.

In [ ]:
BUDGET = 1024 * 28 * 28

def positions(L, t, ov=0.15):
    if t >= L: return [0]
    step = max(1, int(t * (1 - ov)))
    return list(range(0, L - t, step)) + [L - t]

def make_boxes(size):
    W, H = size
    tw = min(W, 900)
    th = min(H, int(BUDGET / tw))
    if th >= H and tw < W: tw = min(W, int(BUDGET / th))
    return [(x, y, x + tw, y + th) for y in positions(H, th) for x in positions(W, tw)]

box_path = f"{CACHE}/zoom_boxes.json"
zoom_cache = json.load(open(box_path)) if os.path.exists(box_path) else {}

@torch.no_grad()
def choose_box(qid, pid):
    key = f"{qid}|{pid}"
    if key in zoom_cache: return tuple(zoom_cache[key])
    img = page_image(pid)
    boxes = make_boxes(img.size)
    if len(boxes) == 1:
        best = boxes[0]
    else:
        embs = embed_images([img.crop(b) for b in boxes])
        qe = q_emb[qid].to("cuda")
        sc = [(qe @ t.to("cuda").T).max(1).values.float().sum().item() for t in embs]
        best = boxes[int(np.argmax(sc))]
    zoom_cache[key] = list(best)
    if len(zoom_cache) % 20 == 0: json.dump(zoom_cache, open(box_path, "w"))
    return best

# how many tiles do the retrieved pages have?
n_tiles = [len(make_boxes(page_image(vis_rank[q["qid"]][0]).size)) for q in q_sub]
print(f"{SET_NAME}: tiles per top-1 page: min {min(n_tiles)}, median {int(np.median(n_tiles))}, max {max(n_tiles)}")
print("example page size:", page_image(vis_rank[q_sub[0]["qid"]][0]).size)

InfoVQA: tiles per top-1 page: min 1, median 4, max 96
example page size: (1024, 1448)


## 4. Run the zoom systems

In [ ]:
def zoom_inputs(q, source="top1", with_page=True):
    pid = vis_rank[q["qid"]][0] if source == "top1" else gold_page(q)
    img = page_image(pid)
    box = choose_box(q["qid"], pid)
    if box == (0, 0, img.size[0], img.size[1]):        # single-tile page: no zoom possible
        return {"images": [img]}
    tile = img.crop(box)
    return {"images": [tile, img] if with_page else [tile]}

accs4 = {}
def run4(tag, **kw):
    _, accs4[tag] = run_system(model, proc, tag, lambda q: zoom_inputs(q, **kw))
    json.dump(zoom_cache, open(box_path, "w"))
    json.dump(accs4, open(f"{CACHE}/acc_task4_zoom.json", "w"), indent=2)

run4("IMP-zoom-top1",     source="top1",   with_page=True)
run4("IMP-zoomonly-top1", source="top1",   with_page=False)
run4("IMP-zoom-oracle",   source="gold",   with_page=True)

IMP-zoom-top1: accuracy = 0.630
IMP-zoomonly-top1: accuracy = 0.540
IMP-zoom-oracle: accuracy = 0.750


## 5. Paired comparisons (same questions)

In [ ]:
def paired_bootstrap(tagA, tagB, n=5000, seed=0):
    pa, pb = load_preds(tagA), load_preds(tagB)
    d = np.array([float(correct(pb[q["qid"]], q["answers"])) - float(correct(pa[q["qid"]], q["answers"]))
                  for q in q_sub])
    rng = np.random.default_rng(seed)
    boots = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    both_wrong = sum(1 for x in d if x == 0)
    print(f"{tagB} - {tagA}: {d.mean():+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
          f"(B wins {int((d>0).sum())}, A wins {int((d<0).sum())}, ties {both_wrong})")

paired_bootstrap("VisRAG-top1", "IMP-hires-top1")
paired_bootstrap("Oracle-image", "Oracle-image-hires")
paired_bootstrap("TextRAG-top1", "VisRAG-top1")

IMP-hires-top1 - VisRAG-top1: +0.065  95% CI [+0.010, +0.120]  (B wins 22, A wins 9, ties 169)
Oracle-image-hires - Oracle-image: +0.090  95% CI [+0.035, +0.150]  (B wins 26, A wins 8, ties 166)
VisRAG-top1 - TextRAG-top1: +0.295  95% CI [+0.215, +0.380]  (B wins 73, A wins 14, ties 113)


In [ ]:
paired_bootstrap("VisRAG-top1", "IMP-zoom-top1")
paired_bootstrap("VisRAG-top1", "IMP-zoomonly-top1")
paired_bootstrap("Oracle-image", "IMP-zoom-oracle")
for t in ["IMP-hires-top1", "Oracle-image-hires"]:
    if os.path.exists(f"{CACHE}/gen_{t}.json"):
        base = "IMP-hires-top1" if t == "IMP-hires-top1" else "Oracle-image-hires"
        paired_bootstrap(base, "IMP-zoom-top1" if t == "IMP-hires-top1" else "IMP-zoom-oracle")

IMP-zoom-top1 - VisRAG-top1: +0.065  95% CI [+0.020, +0.115]  (B wins 17, A wins 4, ties 179)
IMP-zoomonly-top1 - VisRAG-top1: -0.025  95% CI [-0.090, +0.040]  (B wins 18, A wins 23, ties 159)
IMP-zoom-oracle - Oracle-image: +0.085  95% CI [+0.040, +0.135]  (B wins 20, A wins 3, ties 177)
IMP-zoom-top1 - IMP-hires-top1: +0.000  95% CI [-0.045, +0.045]  (B wins 10, A wins 10, ties 180)
IMP-zoom-oracle - Oracle-image-hires: -0.005  95% CI [-0.050, +0.040]  (B wins 11, A wins 12, ties 177)


## 6. Answer-rule analysis (Analysis Questions 2 and 5)
Strict exact match vs a rule that forgives extra words and dropped units but stays strict on numbers and codes, plus a count of degenerate outputs (for example repeated characters).

In [ ]:
UNITS = {"percent", "year", "years", "million", "billion", "thousand", "days", "months", "hours"}

def lenient2(pred, golds):
    p = norm(pred); pt = p.split()
    for g in golds:
        gn = norm(g)
        if not gn: continue
        if gn == p: return True
        gt = gn.split()
        if set(gt) <= set(pt) and len(pt) <= len(gt) + 2: return True       # gold plus up to 2 extra words
        if pt and all(t.isdigit() for t in pt) and set(pt) <= set(gt) and set(gt) - set(pt) <= UNITS:
            return True                                                      # "40" vs "40 percent"
    return False

def degenerate(pred):
    s = pred.replace(" ", "")
    return len(s) >= 8 and len(set(s)) <= 2

rows = {}
for f in sorted(glob.glob(f"{CACHE}/gen_*.json")):
    tag = os.path.basename(f)[4:-5]; pr = json.load(open(f))
    qs = [q for q in q_sub if q["qid"] in pr]
    if len(qs) < len(q_sub): continue
    rows[tag] = {"strict": np.mean([correct(pr[q["qid"]], q["answers"]) for q in qs]),
                 "lenient": np.mean([lenient2(pr[q["qid"]], q["answers"]) for q in qs]),
                 "degenerate": sum(degenerate(pr[q["qid"]]) for q in qs)}
import pandas as pd
print(SET_NAME); print(pd.DataFrame(rows).T.round(3))

InfoVQA
                    strict  lenient  degenerate
IMP-hires-top1       0.630    0.650         0.0
IMP-zoom-oracle      0.750    0.765         0.0
IMP-zoom-top1        0.630    0.645         0.0
IMP-zoomonly-top1    0.540    0.550         0.0
NoRetrieval          0.135    0.155         0.0
Oracle-image-hires   0.755    0.770         0.0
Oracle-image         0.665    0.675         0.0
Oracle-text          0.335    0.360         0.0
TextRAG-top1         0.270    0.280         0.0
TextRAG-top3         0.290    0.300         0.0
VisRAG-top1          0.565    0.575         0.0
VisRAG-top3          0.615    0.635         0.0


In [ ]:
import random, zlib
def rand_inputs(q, with_page=True):
    img = page_image(vis_rank[q["qid"]][0])
    boxes = make_boxes(img.size)
    if len(boxes) == 1: return {"images": [img]}
    box = random.Random(zlib.crc32(q["qid"].encode())).choice(boxes)
    tile = img.crop(box)
    return {"images": [tile, img] if with_page else [tile]}

for tag, wp in [("IMP-randtile-top1", True), ("IMP-randtileonly-top1", False)]:
    _, accs4[tag] = run_system(model, proc, tag, lambda q, wp=wp: rand_inputs(q, wp))

paired_bootstrap("IMP-randtile-top1", "IMP-zoom-top1")
paired_bootstrap("IMP-randtileonly-top1", "IMP-zoomonly-top1")

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


IMP-randtile-top1: accuracy = 0.595
IMP-randtileonly-top1: accuracy = 0.380
IMP-zoom-top1 - IMP-randtile-top1: +0.035  95% CI [+0.000, +0.075]  (B wins 11, A wins 4, ties 185)
IMP-zoomonly-top1 - IMP-randtileonly-top1: +0.160  95% CI [+0.095, +0.230]  (B wins 43, A wins 11, ties 146)


In [ ]:
json.dump(accs4, open(f"{CACHE}/acc_task4_zoom.json", "w"), indent=2)

In [ ]:
paired_bootstrap("VisRAG-top1", "IMP-randtile-top1")

IMP-randtile-top1 - VisRAG-top1: +0.030  95% CI [-0.010, +0.070]  (B wins 11, A wins 5, ties 184)
